# Configure Fabric Workspace

This notebook configures a Microsoft Fabric workspace for the benchmarking solution:

1. **Create a cloud connection** (manual step) — you must create this via the Power BI gateway management page before running this notebook
2. **Update variable library** — writes the connection GUID and notebook GUIDs into the variable library for pipeline orchestration
3. **Update semantic model** — repoints the Power BI semantic model to the lakehouse in the current workspace

## Prerequisites: Create a Cloud Connection

Before running this notebook, you must manually create a shared cloud connection:

1. Go to [Power BI Gateway Management](https://app.powerbi.com/groups/me/gateways)
2. Click **+ New** to create a new connection
3. Set the connection type to **Fabric Data Pipelines**
4. Name it something descriptive (e.g. "Fabric Data Pipelines - Benchmark")
5. Complete the creation wizard
6. Once created, copy the **Connection ID** (GUID) from the connection details
7. Paste the GUID into the `CONNECTION_ID` variable in the next cell

This notebook runs **locally** (outside Fabric) using `azure-identity` for interactive browser authentication.
Set the `WORKSPACE_NAME` and `CONNECTION_ID` variables in the next cell before running.

In [ ]:
import logging

from fabric_admin import (
    FabricRestClient,
    FabricVariableLibrary,
    FabricWorkspace,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

In [ ]:
# === CONFIGURATION ===
# Set the workspace name to configure
WORKSPACE_NAME = "fabric_performance_benchmark_workspace"

# Paste the Connection ID (GUID) from the cloud connection you created manually
# via https://app.powerbi.com/groups/me/gateways
CONNECTION_ID = ""  # e.g. "12dfc5c1-8a87-4c8a-84bc-cf1d1984a6e1"

# Variable library settings
VARIABLE_LIBRARY_NAME = "benchmark_1_variables"
NOTEBOOK_NAMES = ["pyspark_benchmark", "polars_benchmark", "duckdb_benchmark", "pandas_benchmark"]

# Lakehouse and semantic model names (for repointing the Power BI data source)
LAKEHOUSE_NAME = "fabric_performance_benchmark_lakehouse"
SEMANTIC_MODEL_NAME = "benchmark_analytics"

assert CONNECTION_ID, "Please set CONNECTION_ID to the GUID of your cloud connection before running."

## Authenticate and resolve workspace

In [19]:
# Authenticate via interactive browser login and resolve the workspace
client = FabricRestClient()
workspace = FabricWorkspace(client, WORKSPACE_NAME)

2026-04-14 10:21:14,814 fabric_admin._client INFO No token supplied — acquiring token via interactive browser login
2026-04-14 10:21:14,832 azure.core.pipeline.policies.http_logging_policy INFO Request URL: 'https://login.microsoftonline.com/organizations/v2.0/.well-known/openid-configuration'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.1 Python/3.13.12 (Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39)'
No body was attached to the request
2026-04-14 10:21:15,278 azure.core.pipeline.policies.http_logging_policy INFO Response status: 200
Response headers:
    'Cache-Control': 'max-age=86400, private'
    'Content-Type': 'application/json; charset=utf-8'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Access-Control-Allow-Origin': 'REDACTED'
    'Access-Control-Allow-Methods': 'REDACTED'
    'P3P': 'REDACTED'
    'x-ms-request-id': 'd116addc-e3a1-4cf0-bc21-20694fed3400'
    'x-ms-ests-server': 

In [20]:
workspace_id = workspace.workspace_id

2026-04-14 10:21:34,492 fabric_admin.workspace INFO Resolving workspace ID for 'fabric_performance_benchmark_workspace'
2026-04-14 10:21:35,178 fabric_admin.workspace INFO Resolved workspace 'fabric_performance_benchmark_workspace' → 23d2362b-6b5f-4894-8472-c09991fc07a6


## Update variable library

Writes the connection GUID (from the manually-created cloud connection) and each
benchmark notebook's GUID into the variable library so that pipelines can reference them.

In [ ]:
var_lib = FabricVariableLibrary(client, workspace_id)
library_id = var_lib.find_library(VARIABLE_LIBRARY_NAME)

# Build variable updates: connection GUID + notebook GUIDs
variable_updates = {
    "workspace_id": workspace_id,
    "execute_pipeline_connection_id": CONNECTION_ID,
}

for notebook_name in NOTEBOOK_NAMES:
    notebook_id = workspace.get_item_id(notebook_name, item_type="Notebook")
    variable_updates[f"{notebook_name}_notebook_id"] = notebook_id

print("Variables to update:", list(variable_updates.keys()))
var_lib.update_variables(library_id, variable_updates)

2026-04-14 10:21:36,904 fabric_admin.workspace INFO Looking up Notebook 'pyspark_benchmark' in workspace 23d2362b-6b5f-4894-8472-c09991fc07a6



Updating variable library with notebook ID for 'pyspark_benchmark'...


2026-04-14 10:21:37,243 fabric_admin.workspace INFO Found Notebook 'pyspark_benchmark' → 3a4f55e4-c646-42aa-825c-e8b4a44524dd
2026-04-14 10:21:37,246 fabric_admin.variable_library INFO Looking up variable library 'benchmark_1_variables'
2026-04-14 10:21:37,570 fabric_admin.variable_library INFO Found variable library 'benchmark_1_variables' → d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:21:37,571 fabric_admin.variable_library INFO Fetching definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:21:59,692 fabric_admin.variable_library INFO Decoded variables.json — 8 variables found
2026-04-14 10:21:59,694 fabric_admin.variable_library INFO Variable 'pyspark_benchmark_notebook_id' updated: '3a4f55e4-c646-42aa-825c-e8b4a44524dd' → '3a4f55e4-c646-42aa-825c-e8b4a44524dd'
2026-04-14 10:21:59,695 fabric_admin.variable_library INFO Updating definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:22:22,500 fabric_admin.variable_library


Updating variable library with notebook ID for 'polars_benchmark'...


2026-04-14 10:22:23,050 fabric_admin.workspace INFO Found Notebook 'polars_benchmark' → 5204a490-7561-4d6a-a431-50e2511a08bb
2026-04-14 10:22:23,052 fabric_admin.variable_library INFO Looking up variable library 'benchmark_1_variables'
2026-04-14 10:22:23,908 fabric_admin.variable_library INFO Found variable library 'benchmark_1_variables' → d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:22:23,910 fabric_admin.variable_library INFO Fetching definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:22:46,305 fabric_admin.variable_library INFO Decoded variables.json — 8 variables found
2026-04-14 10:22:46,308 fabric_admin.variable_library INFO Variable 'polars_benchmark_notebook_id' updated: '5204a490-7561-4d6a-a431-50e2511a08bb' → '5204a490-7561-4d6a-a431-50e2511a08bb'
2026-04-14 10:22:46,309 fabric_admin.variable_library INFO Updating definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:23:08,852 fabric_admin.variable_library I


Updating variable library with notebook ID for 'duckdb_benchmark'...


2026-04-14 10:23:09,165 fabric_admin.workspace INFO Found Notebook 'duckdb_benchmark' → e8daef53-1e77-45da-8161-a411758a56a3
2026-04-14 10:23:09,167 fabric_admin.variable_library INFO Looking up variable library 'benchmark_1_variables'
2026-04-14 10:23:09,597 fabric_admin.variable_library INFO Found variable library 'benchmark_1_variables' → d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:23:09,598 fabric_admin.variable_library INFO Fetching definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:23:30,940 fabric_admin.variable_library INFO Decoded variables.json — 8 variables found
2026-04-14 10:23:30,942 fabric_admin.variable_library INFO Variable 'duckdb_benchmark_notebook_id' updated: 'e8daef53-1e77-45da-8161-a411758a56a3' → 'e8daef53-1e77-45da-8161-a411758a56a3'
2026-04-14 10:23:30,943 fabric_admin.variable_library INFO Updating definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:23:53,940 fabric_admin.variable_library I


Updating variable library with notebook ID for 'pandas_benchmark'...


2026-04-14 10:23:54,216 fabric_admin.workspace INFO Found Notebook 'pandas_benchmark' → 9e895c1f-7deb-42d8-93af-e01297ea7510
2026-04-14 10:23:54,218 fabric_admin.variable_library INFO Looking up variable library 'benchmark_1_variables'
2026-04-14 10:23:54,652 fabric_admin.variable_library INFO Found variable library 'benchmark_1_variables' → d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:23:54,653 fabric_admin.variable_library INFO Fetching definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:24:15,980 fabric_admin.variable_library INFO Decoded variables.json — 8 variables found
2026-04-14 10:24:15,982 fabric_admin.variable_library INFO Variable 'pandas_benchmark_notebook_id' updated: '9e895c1f-7deb-42d8-93af-e01297ea7510' → '9e895c1f-7deb-42d8-93af-e01297ea7510'
2026-04-14 10:24:15,983 fabric_admin.variable_library INFO Updating definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 10:24:39,184 fabric_admin.variable_library I

## Update semantic model

Repoints the Power BI semantic model's `Sql.Database()` connection to the lakehouse
in the current workspace. This is necessary because the semantic model definition in
Git contains the SQL endpoint server name and database ID from the original workspace.

In [ ]:
# Look up the lakehouse SQL endpoint
sql_server, sql_database_id = workspace.get_lakehouse_sql_endpoint(LAKEHOUSE_NAME)
print(f"Lakehouse SQL endpoint: server={sql_server}, database={sql_database_id}")

# Update the semantic model to point to this lakehouse
workspace.update_semantic_model_lakehouse_connection(
    semantic_model_name=SEMANTIC_MODEL_NAME,
    sql_endpoint_server=sql_server,
    sql_endpoint_database_id=sql_database_id,
)